# Competição: quem vai ser draftado na NFL?

**Ciência de Dados · UFF Petrópolis · 2026.2**

Todo ano, os 32 times da NFL escolhem jogadores universitários no **Draft**. Antes disso, os atletas passam pelo *Combine*, uma bateria de testes físicos (corrida de 40 jardas, saltos, supino, agilidade). Aqui você recebe os dados de **2.224 atletas** (2009–2019) e vai construir um modelo que estima a **probabilidade de um atleta ser draftado** (`Drafted = 1`).

| Etapa | O que fazer |
|---|---|
| 1 | Rodar este notebook até o fim da EDA (ela já está pronta) |
| 2 | Implementar o **seu modelo** na seção *Atividade no notebook* |
| 3 | Baixar o arquivo `modelo_SEUNOME.txt` e enviar para o professor |

**Datas:** competição aberta em **01/out** · entrega do modelo até **29/out** · encerramento em **05/nov**.

**Como o modelo é avaliado:** eu treino o seu modelo com estes dados e o aplico a **557 atletas que você não vê** (teste oculto). A métrica é a **AUC** (Aula 6). Toda semana rodo os modelos recebidos e divulgo o ranking, calculado com metade do teste oculto; o **ranking final** usa a outra metade, então ajustar o modelo ao ranking semanal não adianta. Vale o último arquivo que você enviar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score

sns.set_style("whitegrid")
LARANJA, VERDE, AZUL = "#EB6834", "#1BAF7A", "#2A78D6"  # não draftado, draftado, neutro

## 1. Carregando os dados

Os dados chegam por link: nada para baixar nem subir no Colab.

In [ ]:
# Link do arquivo treino.csv (o professor fornece)
URL_TREINO = "https://raw.githubusercontent.com/HugoHensoldt/nfl/main/treino.csv"

assert not URL_TREINO.startswith("COLE"), "Cole o link do treino.csv na variável URL_TREINO"
treino = pd.read_csv(URL_TREINO)
print(treino.shape)
treino.head()

### Dicionário de dados

| Coluna | Tipo | Unidade | O que é |
|---|---|---|---|
| `Id` | inteiro | – | identificador anônimo do atleta (não é uma característica dele) |
| `Year` | inteiro | ano | ano do Draft |
| `Age` | número | anos | idade no Combine |
| `School` | texto | – | universidade (muitos níveis) |
| `Height` | número | m | altura |
| `Weight` | número | kg | peso |
| `Sprint_40yd` | número | s | tempo na corrida de 40 jardas (**menor é melhor**) |
| `Vertical_Jump` | número | cm | salto vertical |
| `Bench_Press_Reps` | número | repetições | repetições no supino (225 lb) |
| `Broad_Jump` | número | cm | salto horizontal |
| `Agility_3cone` | número | s | teste de agilidade 3-cone (**menor é melhor**) |
| `Shuttle` | número | s | teste de agilidade shuttle (**menor é melhor**) |
| `Player_Type` | texto | – | ataque (`offense`) ou defesa (`defense`) |
| `Position_Type` | texto | – | grupo de posições (linha ofensiva, defensive back…) |
| `Position` | texto | – | posição (QB, WR, OT, CB…) |
| **`Drafted`** | 0 ou 1 | – | **alvo:** 1 = foi draftado |

In [ ]:
treino.info()

## 2. EDA: perguntas, gráficos e conclusões

### Quantos atletas foram draftados?

In [ ]:
contagem = treino["Drafted"].value_counts().sort_index()
pct = 100 * contagem / len(treino)

fig, ax = plt.subplots(figsize=(4.8, 3.2))
barras = ax.bar(["Não draftado", "Draftado"], contagem.values, color=[LARANJA, VERDE])
for b, v, p in zip(barras, contagem.values, pct.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 20, f"{v} ({p:.1f}%)", ha="center")
ax.set_ylim(0, contagem.max() * 1.15)
ax.set_ylabel("atletas")
ax.set_title("Balanceamento do alvo")
plt.show()

### Quais colunas têm valores faltantes, e faltar já indica alguma coisa?

In [ ]:
faltantes = treino.drop(columns="Drafted").isnull().mean().mul(100)
faltantes = faltantes[faltantes > 0].sort_values()

ax = faltantes.plot.barh(color=AZUL, figsize=(6, 3.2))
ax.set_xlabel("% de linhas faltantes")
ax.set_title("Valores faltantes por coluna")
plt.show()

In [ ]:
# Taxa de draft entre quem tem e quem NÃO tem o valor
tabela = pd.DataFrame({
    "% faltante": faltantes,
    "% draftados quando FALTA": [100 * treino.loc[treino[c].isnull(), "Drafted"].mean() for c in faltantes.index],
    "% draftados quando existe": [100 * treino.loc[treino[c].notnull(), "Drafted"].mean() for c in faltantes.index],
}).round(1).sort_values("% faltante", ascending=False)
tabela

### As medidas dos draftados são diferentes das dos não draftados?

In [ ]:
numericas = ["Height", "Weight", "Sprint_40yd", "Vertical_Jump",
             "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle"]

fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for ax, col in zip(axes.ravel(), numericas):
    dados = [treino.loc[treino["Drafted"] == k, col].dropna() for k in (0, 1)]
    bp = ax.boxplot(dados, patch_artist=True, widths=0.55,
                    medianprops=dict(color="#333333"), flierprops=dict(markersize=2, alpha=0.5))
    for caixa, cor in zip(bp["boxes"], [LARANJA, VERDE]):
        caixa.set_facecolor(cor)
        caixa.set_alpha(0.75)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Não", "Sim"])
    ax.set_title(col)
    ax.set_xlabel("Draftado")
plt.tight_layout()
plt.show()

### Quais medidas andam juntas?

In [ ]:
corr = treino[numericas + ["Age", "Drafted"]].corr()
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, cbar=False)
plt.title("Correlação entre as variáveis numéricas")
plt.show()

### A posição muda a chance de ser draftado? E o que significa "rápido" em cada posição?

In [ ]:
por_posicao = treino.groupby("Position")["Drafted"].agg(["mean", "count"]).sort_values("mean")
media_geral = treino["Drafted"].mean() * 100

fig, ax = plt.subplots(figsize=(10, 3.5))
cores = [LARANJA if m * 100 < media_geral else VERDE for m in por_posicao["mean"]]
ax.bar(por_posicao.index, por_posicao["mean"] * 100, color=cores)
ax.axhline(media_geral, color="#333333", ls="--", lw=1)
ax.text(-0.4, media_geral + 2, "média geral", ha="left", fontsize=9)
ax.set_ylabel("% draftados")
ax.set_title("Taxa de draft por posição")
plt.show()

In [ ]:
# Mesma medida, perfis muito diferentes: mediana por grupo de posições
(treino.groupby("Position_Type")[["Weight", "Sprint_40yd", "Vertical_Jump", "Bench_Press_Reps"]]
       .median().round(2))

### E a escola, o ano e a idade?

In [ ]:
print(f"{treino['School'].nunique()} escolas diferentes")
por_escola = treino.groupby("School")["Drafted"].agg(["count", "mean"]).sort_values("count", ascending=False)
print(f"{(por_escola['count'] < 5).mean():.0%} das escolas têm menos de 5 atletas")
por_escola.head(10).assign(mean=lambda d: (100 * d["mean"]).round(1)).rename(
    columns={"count": "atletas", "mean": "% draftados"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

por_ano = treino.groupby("Year")["Drafted"].agg(["count", "mean"])
axes[0].bar(por_ano.index, por_ano["count"], color=AZUL)
axes[0].set_title("Atletas por ano do Draft")
axes[0].set_xticks(por_ano.index)
axes[0].tick_params(axis="x", labelrotation=45)
axes[0].set_xlabel("ano")

por_idade = treino.groupby("Age")["Drafted"].agg(["count", "mean"])
por_idade = por_idade[por_idade["count"] >= 20]
axes[1].plot(por_idade.index, por_idade["mean"] * 100, marker="o", color=VERDE)
axes[1].set_xticks(por_idade.index)
axes[1].set_title("% draftados por idade (idades com 20+ atletas)")
axes[1].set_xlabel("idade")
plt.tight_layout()
plt.show()

### O que a EDA sugere

Para você pensar, não é uma receita:

- o **balanceamento** do alvo ajuda a escolher a métrica (já fixada: AUC);
- há **colunas faltantes**, e a tabela acima mostra se o faltante tem relação com o alvo;
- as medidas físicas fazem sentido **dentro de cada posição**, não no conjunto todo;
- `School` tem muitos níveis; `Position`, poucos;
- algumas variáveis são tempos, em que menor é melhor.

## 3. Como entregar o seu modelo

Você envia **um arquivo `.txt`** com código Python que define a função `criar_modelo()`. Ela devolve um **modelo do scikit-learn ainda não treinado** (de preferência um `Pipeline`, com todo o pré-processamento dentro). Eu treino o modelo com o `treino.csv` e avalio no teste oculto.

Regras:

1. `criar_modelo()` não recebe nada e devolve um objeto com `.fit(X, y)` e `.predict_proba(X)`.
2. `X` é um DataFrame com as colunas do treino **menos `Id` e `Drafted`**, com os mesmos nomes. Transformações de colunas (por nome) devem estar dentro do modelo.
3. Use só `numpy`, `pandas` e `scikit-learn`.
4. Fixe `random_state` onde houver aleatoriedade.
5. O treino do seu modelo deve levar **menos de 5 minutos**.
6. O Colab tem scikit-learn mais novo que o do meu computador (**1.0.2**). Evite recursos recentes, como `OneHotEncoder(sparse_output=...)`, `min_frequency`, `set_output` e `TargetEncoder`. Se o modelo der erro lá, ele aparece como erro no ranking.
7. O código não deve ler arquivos nem a internet: só usa o `X` que receber.

In [ ]:
X = treino.drop(columns=["Id", "Drafted"])
y = treino["Drafted"]


def avaliar_modelo(modelo, n_splits=5):
    """AUC em validação cruzada estratificada: use para comparar as suas ideias."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")
    print("AUC por fold:", np.round(aucs, 3))
    print(f"AUC média: {aucs.mean():.4f} (± {aucs.std():.4f})")
    return aucs.mean()

## 4. Atividade no notebook: o seu modelo

A célula abaixo **não executa** o código: o `%%writefile` só grava o conteúdo no arquivo `modelo.txt`. Ela traz um esqueleto mínimo, só para mostrar o formato. **Substitua pelo seu modelo.**

Ideias para explorar (use o que viu na EDA): outras colunas, o tratamento dos faltantes, variáveis categóricas, outros modelos e hiperparâmetros.

Fluxo: edite a célula → rode-a → rode a célula de teste logo abaixo → repita até gostar da AUC.

In [ ]:
%%writefile modelo.txt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

NUMERICAS = ["Age", "Height", "Weight", "Sprint_40yd", "Vertical_Jump",
             "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle"]


def criar_modelo():
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("esc", StandardScaler())]), NUMERICAS),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

### Testando o seu modelo

Esta célula carrega o `modelo.txt` do mesmo jeito que eu farei na avaliação e calcula a AUC por validação cruzada nos dados de treino (o teste oculto fica comigo).

In [ ]:
def carregar_modelo(caminho="modelo.txt"):
    namespace = {}
    exec(compile(open(caminho, encoding="utf-8").read(), caminho, "exec"), namespace)
    return namespace["criar_modelo"]()


modelo = carregar_modelo()
avaliar_modelo(modelo)

# Checagem do formato: treina com tudo e prevê algumas linhas (é isso que a avaliação faz)
modelo.fit(X, y)
prob = modelo.predict_proba(X.head())[:, 1]
print("probabilidades de exemplo:", prob.round(3))

### Baixando o arquivo para enviar

Troque `NOME` pelo seu nome e rode. No Colab, o download começa sozinho.

In [ ]:
import shutil

NOME = "seu_nome"  # exemplo: "dani"

arquivo = f"modelo_{NOME}.txt"
shutil.copy("modelo.txt", arquivo)
try:
    from google.colab import files
    files.download(arquivo)
except ImportError:
    print("Fora do Colab: o arquivo está na pasta do notebook:", arquivo)